In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
sns.set_palette("Set2")

In [ ]:
spark = (
    SparkSession.builder
    .appName("Supply Chain Risk Analysis")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")
    .csv("../data/raw/DataCoSupplyChainDataset.csv")
)

In [ ]:
rows = df.count()
cols = len(df.columns)

print("Rows :", rows)
print("Columns :", cols)

In [ ]:
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
for column in df.columns:
    print(column)

In [ ]:
datatype_df = pd.DataFrame(df.dtypes, columns=["Column", "Datatype"])
datatype_df

In [ ]:
missing = []

total_rows = df.count()

for column in df.columns:

    missing_count = df.filter(col(column).isNull()).count()

    missing_percent = round(
        (missing_count / total_rows) * 100,
        2
    )

    missing.append([
        column,
        missing_count,
        missing_percent
    ])

missing_df = pd.DataFrame(
    missing,
    columns=[
        "Column",
        "Missing Values",
        "Percentage"
    ]
)

missing_df.sort_values(
    by="Missing Values",
    ascending=False
)

In [ ]:
top_missing = (
    missing_df
    .sort_values(
        "Missing Values",
        ascending=False
    )
    .head(15)
)

plt.figure(figsize=(12,6))

sns.barplot(
    data=top_missing,
    x="Missing Values",
    y="Column"
)

plt.title("Top Columns with Missing Values")

plt.show()

In [ ]:
duplicate_count = (
    rows -
    df.dropDuplicates().count()
)

print("Duplicate Rows :", duplicate_count)

In [ ]:
df.describe().show()

In [ ]:
numeric_columns = []

for field in df.schema.fields:

    if isinstance(
        field.dataType,
        (
            IntegerType,
            LongType,
            FloatType,
            DoubleType
        )
    ):
        numeric_columns.append(field.name)

numeric_columns

In [ ]:
df.select(numeric_columns).describe().show()

In [ ]:
late_delivery = (
    df.groupBy("Late_delivery_risk")
    .count()
    .toPandas()
)

late_delivery

In [ ]:
plt.figure(figsize=(6,4))

sns.barplot(
    data=late_delivery,
    x="Late_delivery_risk",
    y="count"
)

plt.title("Late Delivery Risk")

plt.xlabel("Late Delivery")

plt.ylabel("Number of Orders")

plt.show()

In [ ]:
delivery_status = (
    df.groupBy("Delivery Status")
    .count()
    .orderBy(desc("count"))
    .toPandas()
)

delivery_status

In [ ]:
plt.figure(figsize=(12,5))

sns.barplot(
    data=delivery_status,
    x="Delivery Status",
    y="count"
)

plt.xticks(rotation=35)

plt.show()

In [ ]:
customer_segment = (
    df.groupBy("Customer Segment")
    .count()
    .orderBy(desc("count"))
    .toPandas()
)

customer_segment

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=customer_segment,
    x="Customer Segment",
    y="count"
)

plt.title("Customer Segment Distribution")

plt.show()

In [ ]:
shipping_mode = (
    df.groupBy("Shipping Mode")
      .count()
      .orderBy(desc("count"))
      .toPandas()
)

shipping_mode

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=shipping_mode,
    x="Shipping Mode",
    y="count"
)

plt.title("Orders by Shipping Mode")
plt.xticks(rotation=20)

plt.show()

In [ ]:
late_shipping = (
    df.groupBy(
        "Shipping Mode",
        "Late_delivery_risk"
    )
    .count()
    .toPandas()
)

late_shipping

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    data=late_shipping,
    x="Shipping Mode",
    y="count",
    hue="Late_delivery_risk"
)

plt.title("Late Delivery Risk by Shipping Mode")

plt.show()

In [ ]:
market = (
    df.groupBy("Market")
      .count()
      .orderBy(desc("count"))
      .toPandas()
)

market

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    data=market,
    x="Market",
    y="count"
)

plt.xticks(rotation=30)

plt.title("Orders by Market")

plt.show()

In [ ]:
region = (
    df.groupBy("Order Region")
      .count()
      .orderBy(desc("count"))
      .toPandas()
)

region.head()

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=region.head(15),
    x="count",
    y="Order Region"
)

plt.title("Top 15 Regions by Orders")

plt.show()

In [ ]:
country = (
    df.groupBy("Order Country")
      .count()
      .orderBy(desc("count"))
      .limit(15)
      .toPandas()
)

country

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=country,
    x="count",
    y="Order Country"
)

plt.title("Top Countries")

plt.show()

In [ ]:
cities = (
    df.groupBy("Order City")
      .count()
      .orderBy(desc("count"))
      .limit(20)
      .toPandas()
)

cities.head()

In [ ]:
plt.figure(figsize=(12,7))

sns.barplot(
    data=cities,
    x="count",
    y="Order City"
)

plt.title("Top Cities by Orders")

plt.show()

In [ ]:
sales = df.select("Sales").toPandas()

plt.figure(figsize=(10,5))

sns.histplot(
    sales["Sales"],
    bins=50,
    kde=True
)

plt.title("Sales Distribution")

plt.show()

In [ ]:
profit = df.select("Benefit per order").toPandas()

plt.figure(figsize=(10,5))

sns.histplot(
    profit["Benefit per order"],
    bins=50,
    kde=True
)

plt.title("Benefit per Order")

plt.show()

In [ ]:
quantity = df.select("Order Item Quantity").toPandas()

plt.figure(figsize=(10,5))

sns.histplot(
    quantity["Order Item Quantity"],
    bins=20
)

plt.title("Order Quantity Distribution")

plt.show()

In [ ]:
shipping_real = df.select("Days for shipping (real)").toPandas()

plt.figure(figsize=(10,5))

sns.histplot(
    shipping_real["Days for shipping (real)"],
    bins=20
)

plt.title("Actual Shipping Days")

plt.show()

In [ ]:
shipping_schedule = df.select("Days for shipment (scheduled)").toPandas()

plt.figure(figsize=(10,5))

sns.histplot(
    shipping_schedule["Days for shipment (scheduled)"],
    bins=20
)

plt.title("Scheduled Shipping Days")

plt.show()

In [ ]:
df = df.withColumn(
    "Shipping Delay",
    col("Days for shipping (real)") -
    col("Days for shipment (scheduled)")
)

In [ ]:
delay = df.select("Shipping Delay").toPandas()

plt.figure(figsize=(10,5))

sns.histplot(
    delay["Shipping Delay"],
    bins=30,
    kde=True
)

plt.title("Shipping Delay")

plt.show()

In [ ]:
avg_delay = (
    df.groupBy("Shipping Mode")
      .agg(
          avg("Shipping Delay").alias("Average Delay")
      )
      .orderBy(desc("Average Delay"))
      .toPandas()
)

avg_delay

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=avg_delay,
    x="Shipping Mode",
    y="Average Delay"
)

plt.title("Average Shipping Delay")

plt.show()

In [ ]:
category = (
    df.groupBy("Category Name")
      .count()
      .orderBy(desc("count"))
      .limit(15)
      .toPandas()
)

category

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=category,
    x="count",
    y="Category Name"
)

plt.title("Top Product Categories")

plt.show()

In [ ]:
department = (
    df.groupBy("Department Name")
      .count()
      .orderBy(desc("count"))
      .toPandas()
)

department

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=department,
    x="Department Name",
    y="count"
)

plt.xticks(rotation=25)

plt.show()

In [ ]:
segment_late = (
    df.groupBy(
        "Customer Segment",
        "Late_delivery_risk"
    )
    .count()
    .toPandas()
)

segment_late.head()

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=segment_late,
    x="Customer Segment",
    y="count",
    hue="Late_delivery_risk"
)

plt.title("Late Delivery by Customer Segment")

plt.show()

In [ ]:
correlation_columns = [
    "Sales",
    "Benefit per order",
    "Order Item Quantity",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Shipping Delay"
]

corr_df = (
    df.select(correlation_columns)
      .toPandas()
)

corr_df.head()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    corr_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
sales = df.select("Sales").toPandas()

plt.figure(figsize=(10,3))

sns.boxplot(
    x=sales["Sales"]
)

plt.title("Sales Outliers")

plt.show()

In [ ]:
profit = df.select("Benefit per order").toPandas()

plt.figure(figsize=(10,3))

sns.boxplot(
    x=profit["Benefit per order"]
)

plt.title("Benefit per Order Outliers")

plt.show()

In [ ]:
delay = df.select("Shipping Delay").toPandas()

plt.figure(figsize=(10,3))

sns.boxplot(
    x=delay["Shipping Delay"]
)

plt.title("Shipping Delay Outliers")

plt.show()

In [ ]:
category_sales = (
    df.groupBy("Category Name")
      .agg(
          avg("Sales").alias("Average Sales")
      )
      .orderBy(desc("Average Sales"))
      .limit(15)
      .toPandas()
)

category_sales

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=category_sales,
    x="Average Sales",
    y="Category Name"
)

plt.title("Average Sales by Category")

plt.show()

In [ ]:
category_profit = (
    df.groupBy("Category Name")
      .agg(
          avg("Benefit per order").alias("Average Profit")
      )
      .orderBy(desc("Average Profit"))
      .limit(15)
      .toPandas()
)

category_profit

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    data=category_profit,
    x="Average Profit",
    y="Category Name"
)

plt.title("Average Profit by Category")

plt.show()

In [ ]:
late_region = (
    df.groupBy("Order Region")
      .agg(
          avg("Late_delivery_risk").alias("Late Delivery Rate")
      )
      .orderBy(desc("Late Delivery Rate"))
      .toPandas()
)

late_region.head()

In [ ]:
plt.figure(figsize=(12,8))

sns.barplot(
    data=late_region.head(15),
    x="Late Delivery Rate",
    y="Order Region"
)

plt.title("Top Regions with Highest Late Delivery Risk")

plt.show()

In [ ]:
late_category = (
    df.groupBy("Category Name")
      .agg(
          avg("Late_delivery_risk").alias("Late Delivery Rate")
      )
      .orderBy(desc("Late Delivery Rate"))
      .limit(15)
      .toPandas()
)

late_category